In [3]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from sklearn.utils import class_weight

In [4]:
DATASET_PATH = '/kaggle/input/skin-diseases-image-dataset/IMG_CLASSES' 
IMG_SIZE = 224 # EfficientNetB0 expects 224x224
BATCH_SIZE = 32
CHANNELS = 3
NUM_CLASSES = 10

In [5]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

class_names = train_ds.class_names

Found 27153 files belonging to 10 classes.
Using 21723 files for training.


I0000 00:00:1769498327.149611     319 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1769498327.153504     319 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 27153 files belonging to 10 classes.
Using 5430 files for validation.


In [6]:
y_train = np.concatenate([y for x, y in train_ds], axis=0)
y_train_indices = np.argmax(y_train, axis=1)

weights = class_weight.compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train_indices),
    y=y_train_indices
)
class_weights_dict = dict(enumerate(weights))

print(f"Calculated Class Weights: {class_weights_dict}")

Calculated Class Weights: {0: np.float64(1.5833090379008747), 1: np.float64(1.2838652482269504), 2: np.float64(0.8685725709716113), 3: np.float64(2.1766533066132263), 4: np.float64(0.8120747663551402), 5: np.float64(0.3418791312559018), 6: np.float64(1.328623853211009), 7: np.float64(1.307826610475617), 8: np.float64(1.4737449118046133), 9: np.float64(1.596105804555474)}


In [7]:
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.2),
    layers.RandomZoom(0.2),
])

In [8]:
base_model = tf.keras.applications.EfficientNetB0(
    include_top=False, 
    weights='imagenet', 
    input_shape=(IMG_SIZE, IMG_SIZE, CHANNELS)
)

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [9]:
base_model.trainable = False

model = models.Sequential([
    layers.Input(shape=(IMG_SIZE, IMG_SIZE, CHANNELS)),
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

In [10]:
print("Starting initial training of the head...")
history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights_dict # Applying the weights here
)

Starting initial training of the head...
Epoch 1/10


E0000 00:00:1769498399.196324     319 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/sequential_1_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer
I0000 00:00:1769498401.900359     388 cuda_dnn.cc:529] Loaded cuDNN version 91002


679/679 ━━━━━━━━━━━━━━━━━━━━ 87s 108ms/step - accuracy: 0.4273 - auc: 0.8408 - loss: 1.9559 - val_accuracy: 0.6366 - val_auc: 0.9444 - val_loss: 0.9904
Epoch 2/10
679/679 ━━━━━━━━━━━━━━━━━━━━ 66s 97ms/step - accuracy: 0.5733 - auc: 0.9248 - loss: 1.3527 - val_accuracy: 0.6457 - val_auc: 0.9486 - val_loss: 0.9548
Epoch 3/10
679/679 ━━━━━━━━━━━━━━━━━━━━ 66s 97ms/step - accuracy: 0.5948 - auc: 0.9335 - loss: 1.2621 - val_accuracy: 0.6685 - val_auc: 0.9519 - val_loss: 0.9178
Epoch 4/10
679/679 ━━━━━━━━━━━━━━━━━━━━ 66s 97ms/step - accuracy: 0.6037 - auc: 0.9370 - loss: 1.2267 - val_accuracy: 0.6810 - val_auc: 0.9551 - val_loss: 0.8823
Epoch 5/10
679/679 ━━━━━━━━━━━━━━━━━━━━ 68s 99ms/step - accuracy: 0.6092 - auc: 0.9377 - loss: 1.2172 - val_accuracy: 0.6808 - val_auc: 0.9547 - val_loss: 0.8861
Epoch 6/10
679/679 ━━━━━━━━━━━━━━━━━━━━ 66s 98ms/step - accuracy: 0.6136 - auc: 0.9389 - loss: 1.2092 - val_accuracy: 0.6735 - val_auc: 0.9550 - val_loss: 0.8876
Epoch 7/10
679/679 ━━━━━━━━━━━━━━━━━━━

In [11]:
print("Unfreezing base model for fine-tuning...")
base_model.trainable = True

Unfreezing base model for fine-tuning...


In [13]:
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-5), 
    loss='categorical_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    class_weight=class_weights_dict
)

Epoch 1/10


E0000 00:00:1769501690.732633     319 meta_optimizer.cc:967] layout failed: INVALID_ARGUMENT: Size of values 0 does not match size of permutation 4 @ fanin shape inStatefulPartitionedCall/sequential_1_1/efficientnetb0_1/block2b_drop_1/stateless_dropout/SelectV2-2-TransposeNHWCToNCHW-LayoutOptimizer


679/679 ━━━━━━━━━━━━━━━━━━━━ 289s 374ms/step - accuracy: 0.6798 - auc: 0.9583 - loss: 1.0076 - val_accuracy: 0.6797 - val_auc: 0.9577 - val_loss: 0.8626
Epoch 2/10
679/679 ━━━━━━━━━━━━━━━━━━━━ 251s 370ms/step - accuracy: 0.6913 - auc: 0.9606 - loss: 0.9673 - val_accuracy: 0.6834 - val_auc: 0.9591 - val_loss: 0.8502
Epoch 3/10
679/679 ━━━━━━━━━━━━━━━━━━━━ 251s 370ms/step - accuracy: 0.6984 - auc: 0.9634 - loss: 0.9416 - val_accuracy: 0.6926 - val_auc: 0.9607 - val_loss: 0.8308
Epoch 4/10
679/679 ━━━━━━━━━━━━━━━━━━━━ 252s 371ms/step - accuracy: 0.7012 - auc: 0.9647 - loss: 0.9244 - val_accuracy: 0.6991 - val_auc: 0.9623 - val_loss: 0.8141
Epoch 5/10
679/679 ━━━━━━━━━━━━━━━━━━━━ 250s 368ms/step - accuracy: 0.7118 - auc: 0.9660 - loss: 0.9000 - val_accuracy: 0.7020 - val_auc: 0.9635 - val_loss: 0.7995
Epoch 6/10
679/679 ━━━━━━━━━━━━━━━━━━━━ 251s 370ms/step - accuracy: 0.7181 - auc: 0.9679 - loss: 0.8769 - val_accuracy: 0.7064 - val_auc: 0.9633 - val_loss: 0.7991
Epoch 7/10
679/679 ━━━━━━━━

In [14]:
model.save("efficientB0_model.keras")